# HorizonRev TRL PPO (Direct Colab)

Paste your GitHub repo URL in the first setup cell, run all cells, and you will get:
- PPO reward curve
- random vs trained comparison
- minimal vs structured report comparison in capped/uncapped modes

In [ ]:
REPO_URL = "https://github.com/sivagirish81/HorizonRev.git"  # replace me
!git clone $REPO_URL horizonrev_repo
%cd horizonrev_repo

# Pin TRL-compatible versions for this notebook API.
!pip install -q --upgrade pip
!pip install -q "trl==0.11.4" "transformers==4.43.4" "accelerate==0.33.0" "peft==0.12.0" "torch"
!pip install -q -e .

print("Install complete. If imports fail, Runtime -> Restart runtime, then re-run from imports cell.")

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from transformers import AutoTokenizer
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from horizonrev import HorizonRevEnv, HorizonRevConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
ACTIONS = list(range(8))
ACTION_TOKENS = [f'<A{i}>' for i in ACTIONS]

def obs_to_text(obs):
    # Includes churn-sensitive cohort signals (pct_young, avg_quality).
    return (
        f"month={obs[0]:.3f} arr={obs[1]:.3f} conv={obs[2]:.3f} churn={obs[3]:.3f} "
        f"discount={obs[4]:.3f} smb_dem={obs[5]:.3f} ent_dem={obs[6]:.3f} "
        f"pricing={int(obs[7])} onboarding={int(obs[8])} focus={int(obs[9])} "
        f"pct_young={obs[10]:.3f} avg_quality={obs[11]:.3f}. "
        f"Emit one action token from: {' '.join(ACTION_TOKENS)}"
    )

def decode_action(text):
    for i, tok in enumerate(ACTION_TOKENS):
        if tok in text:
            return i
    return random.randint(0, 7)

def minimal_report(_obs):
    return 'Action selected.'

def structured_report(obs):
    month = int(round(float(obs[0]) * 6))
    return (
        f"Hypothesis: month {month} plan can improve ARR while controlling churn.\n"
        "Action: pick discount/onboarding/sales focus based on current metrics.\n"
        "Expected Impact: improve conversion and protect delayed churn and cohort quality.\n"
        "Risks: over-discounting can backfire under drift and raise delayed churn.\n"
        "Next Step: track ARR, churn, conversion, discount, drift, month, pct_young, and avg_quality."
    )

def make_config(reward_mode='uncapped'):
    return HorizonRevConfig(**{**HorizonRevConfig.default().__dict__, 'reward_mode': reward_mode})

def evaluate(agent='random', episodes=20, model=None, tokenizer=None, reward_mode='uncapped', report_style='structured'):
    scores = []
    for ep in range(episodes):
        env = HorizonRevEnv(make_config(reward_mode))
        obs = env.reset(seed=1000 + ep)
        done = False
        total = 0.0
        while not done:
            if agent == 'random':
                action = random.randint(0, 7)
            else:
                q = tokenizer(obs_to_text(obs), return_tensors='pt').input_ids.to(model.pretrained_model.device)
                with torch.no_grad():
                    r = model.generate(q, max_new_tokens=4, do_sample=True, top_k=0, top_p=1.0)
                action = decode_action(tokenizer.decode(r[0][q.shape[-1]:], skip_special_tokens=False))
            report = structured_report(obs) if report_style == 'structured' else minimal_report(obs)
            obs, reward, done, _ = env.step(action, agent_report=report)
            total += reward
        scores.append(total)
    return float(np.mean(scores)), scores

In [ ]:
model_name = 'sshleifer/tiny-gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({'additional_special_tokens': ACTION_TOKENS})
model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name)
model.pretrained_model.resize_token_embeddings(len(tokenizer))
model.to(device)

ppo_trainer = PPOTrainer(
    config=PPOConfig(batch_size=4, mini_batch_size=2, learning_rate=1e-5, ppo_epochs=2, log_with=None),
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
)

In [ ]:
reward_curve = []
for ep in range(40):
    env = HorizonRevEnv(make_config('uncapped'))
    obs = env.reset(seed=ep)
    done = False
    total = 0.0
    qts, rts, rs = [], [], []
    while not done:
        q = tokenizer(obs_to_text(obs), return_tensors='pt').input_ids.to(model.pretrained_model.device)
        r = ppo_trainer.generate(q, max_new_tokens=4, do_sample=True, top_k=0, top_p=1.0)
        a = decode_action(tokenizer.decode(r[0][q.shape[-1]:], skip_special_tokens=False))
        obs, reward, done, _ = env.step(a, agent_report=structured_report(obs))
        total += reward
        qts.append(q.squeeze(0))
        rts.append(r[0][q.shape[-1]:])
        rs.append(torch.tensor(reward, dtype=torch.float32).to(model.pretrained_model.device))
    ppo_trainer.step(qts, rts, rs)
    reward_curve.append(total)

plt.figure(figsize=(7, 3.5))
plt.plot(reward_curve)
plt.title('PPO reward curve (uncapped mode)')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.show()

In [ ]:
random_avg, _ = evaluate(agent='random', episodes=20, reward_mode='uncapped', report_style='minimal')
trained_avg, _ = evaluate(agent='trained', episodes=20, model=model, tokenizer=tokenizer, reward_mode='uncapped', report_style='structured')
print(f'Random avg reward (minimal report, uncapped): {random_avg:.3f}')
print(f'Trained avg reward (structured report, uncapped): {trained_avg:.3f}')
print(f'Improvement: {trained_avg-random_avg:+.3f}')

uncapped_structured, _ = evaluate(agent='trained', episodes=20, model=model, tokenizer=tokenizer, reward_mode='uncapped', report_style='structured')
uncapped_minimal, _ = evaluate(agent='trained', episodes=20, model=model, tokenizer=tokenizer, reward_mode='uncapped', report_style='minimal')
capped_structured, _ = evaluate(agent='trained', episodes=20, model=model, tokenizer=tokenizer, reward_mode='capped', report_style='structured')
print(f'Uncapped + structured: {uncapped_structured:.3f}')
print(f'Uncapped + minimal:    {uncapped_minimal:.3f}')
print(f'Capped + structured:   {capped_structured:.3f}')

In [ ]:
torch.save(model.state_dict(), 'horizonrev_trl_policy.pt')
print('Saved model to horizonrev_trl_policy.pt')